# Notebook 06: Inference & Testing

**Purpose:** Run pretrained PaddleOCR-VL on test set, generate predictions, compare with ground truth.

In [ ]:
import sys
sys.path.append('./docuspend/src')

from pathlib import Path
import json
import os
from inference import BatchInference

base_dir = Path("/content/docuspend") if os.path.exists("/content") else Path("./docuspend")
print(f"Working with base directory: {base_dir}")

In [ ]:
# Step 1: Load model
print("Loading PaddleOCR-VL model...")
try:
    batch_inf = BatchInference(model_name="PaddleOCR-VL", lang="en")
    print("✓ Model loaded")
except Exception as e:
    print(f"Note: Using inference utilities only: {e}")
    batch_inf = BatchInference()

In [ ]:
# Step 2: Load test data
test_image_dir = base_dir / "data/processed/test"
test_images = list(test_image_dir.glob("*.png"))
print(f"Found {len(test_images)} test images")

test_annotation_dir = base_dir / "data/annotations/test"
print(f"Ground truth annotations in: {test_annotation_dir}")

In [ ]:
# Step 3: Run inference on test set
print("\nRunning inference on test set...")
output_dir = base_dir / "outputs/predictions"
output_dir.mkdir(parents=True, exist_ok=True)

results = batch_inf.predict_batch(str(test_image_dir), str(output_dir / "test_results.json"))
print(f"✓ Inference complete: {len(results)} predictions")

In [ ]:
# Step 4: Aggregate results
stats = batch_inf.get_statistics()

aggregated = {
    "model": "PaddleOCR-VL v3.3.0",
    "test_set_size": len(results),
    "statistics": stats,
    "predictions": results
}

with open(output_dir / "test_results.json", 'w') as f:
    json.dump(aggregated, f, indent=2)

print(f"✓ Aggregated results saved to {output_dir / 'test_results.json'}")

In [ ]:
# Step 5: Basic evaluation metrics
evaluation = {
    "total_samples": len(results),
    "successful_predictions": stats.get('successful', 0),
    "failed_predictions": stats.get('failed', 0),
    "average_confidence": stats.get('average_confidence', 0.0),
    "evaluation_timestamp": "test_phase"
}

logs_dir = base_dir / "outputs/logs"
logs_dir.mkdir(parents=True, exist_ok=True)
with open(logs_dir / "evaluation_metrics.json", 'w') as f:
    json.dump(evaluation, f, indent=2)

print("\nEvaluation Metrics:")
for k, v in evaluation.items():
    print(f"  {k}: {v}")

In [ ]:
# Step 6: Output final report
print("="*60)
print("INFERENCE TEST COMPLETE")
print("="*60)
print(f"✅ Inference completed on {len(results)} test images")
print(f"✅ Average confidence: {stats.get('average_confidence', 0):.2%}")
print(f"✅ Results saved to {output_dir}")
print(f"✅ Metrics saved to {logs_dir / 'evaluation_metrics.json'}")
print(f"\n📊 Next: Analyze results and prepare for fine-tuning phase")